In [ ]:
# Attention LSTM Seq2Seq for Nepali Text Summarization
#
# Installation (run once):
#   pip install torch pandas numpy scikit-learn sentencepiece \
#               sacrebleu rouge-score bert-score tqdm
#
# Data:    small.csv, medium.csv, large.csv in $NEPALI_DATA_DIR or ./data/.
# Outputs: $NEPALI_WORK_DIR (default: ./lstm_attention_workspace).
# Tokenizer: $NEPALI_TOKENIZER, or trained from the train split.
# Requires CUDA GPU.

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from tqdm.auto import tqdm


CONFIG = {
    "data_dir":  Path(os.environ.get("NEPALI_DATA_DIR", "./data")),
    "work_dir":  Path(os.environ.get("NEPALI_WORK_DIR", "./lstm_attention_workspace")),
    "tokenizer": os.environ.get("NEPALI_TOKENIZER"),

    # 80:10:10 split
    "test_size":       0.2,
    "seed":            42,
    "max_content_len": 768,
    "max_summary_len": 100,
    "spm_vocab_size":  32000,

    # Model architecture
    "embedding_dim": 256,
    "hidden_dim":    512,
    "num_layers":    2,
    "dropout":       0.3,

    # Training
    "batch_size":      32,
    "epochs":          30,
    "lr":              1e-3,
    "accum_steps":     1,
    "grad_clip":       1.0,
    "teacher_forcing": 0.5,
}


# Reproducibility and runtime
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("CUDA GPU required.")

WORK_DIR     = CONFIG["work_dir"]
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH    = WORK_DIR / "lstm_attention_best.pt"
TEST_RAW_CSV = WORK_DIR / "lstm_attention_test_raw.csv"
PRED_CSV     = WORK_DIR / "lstm_attention_test_predictions.csv"
SCORES_CSV   = WORK_DIR / "lstm_attention_final_scores.csv"


# Load, clean, split 80/10/10
def load_and_split(cfg):
    data_dir = cfg["data_dir"]
    required = ("small.csv", "medium.csv", "large.csv")
    missing = [n for n in required if not (data_dir / n).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing {missing} in {data_dir.resolve()}. "
            f"Set NEPALI_DATA_DIR or place the CSVs in ./data/."
        )
    df = pd.concat([pd.read_csv(data_dir / n) for n in required], ignore_index=True)
    df = df.dropna(subset=["content", "summary"]).copy()
    df["content"] = df["content"].astype(str).str.strip()
    df["summary"] = df["summary"].astype(str).str.strip()
    df = df[(df["content"] != "") & (df["summary"] != "")]
    df = df.drop_duplicates(subset=["content"], keep="first").reset_index(drop=True)

    train_df, temp_df = train_test_split(
        df, test_size=cfg["test_size"], random_state=cfg["seed"], shuffle=True
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=cfg["seed"], shuffle=True
    )
    return train_df, val_df, test_df


train_df, val_df, test_df = load_and_split(CONFIG)
test_df[["content", "summary"]].to_csv(TEST_RAW_CSV, index=False, encoding="utf-8")


# SentencePiece tokenizer: reuse, or train from the train split
def build_or_load_tokenizer(cfg, train_frame):
    candidates = []
    if cfg["tokenizer"]:
        candidates.append(Path(cfg["tokenizer"]))
    candidates.append(WORK_DIR / "nepali_sp.model")
    for path in candidates:
        if path.is_file():
            return spm.SentencePieceProcessor(model_file=str(path))

    corpus_path = WORK_DIR / "corpus.txt"
    with corpus_path.open("w", encoding="utf-8") as f:
        for t in train_frame["content"].astype(str):
            f.write(t.strip() + "\n")
        for t in train_frame["summary"].astype(str):
            f.write(t.strip() + "\n")

    # Dynamic max_sentence_length: longest line in bytes + safe buffer
    max_bytes = 0
    with corpus_path.open("r", encoding="utf-8") as f:
        for line in f:
            max_bytes = max(max_bytes, len(line.strip().encode("utf-8")))
    safe_len = max(max_bytes + 1000, 10000)

    model_prefix = WORK_DIR / "nepali_sp"
    spm.SentencePieceTrainer.train(
        input=str(corpus_path),
        model_prefix=str(model_prefix),
        vocab_size=cfg["spm_vocab_size"],
        model_type="unigram",
        character_coverage=1.0,
        max_sentence_length=safe_len,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    )
    corpus_path.unlink()
    return spm.SentencePieceProcessor(model_file=f"{model_prefix}.model")


sp = build_or_load_tokenizer(CONFIG, train_df)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.unk_id(), sp.bos_id(), sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()


# Encode all splits to token ids
def encode(df, sp_proc, cfg):
    df = df.copy()
    df["content_ids"] = df["content"].astype(str).apply(
        lambda x: sp_proc.encode(x, out_type=int)[:cfg["max_content_len"]]
    )
    df["summary_ids"] = df["summary"].astype(str).apply(
        lambda x: [BOS_ID]
        + sp_proc.encode(x, out_type=int)[:cfg["max_summary_len"] - 2]
        + [EOS_ID]
    )
    return df


train_df, val_df, test_df = (encode(d, sp, CONFIG) for d in (train_df, val_df, test_df))


# Dataset and collator
class SummarizationDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        return {
            "content": torch.tensor(row["content_ids"], dtype=torch.long),
            "summary": torch.tensor(row["summary_ids"], dtype=torch.long),
        }


def collate_fn(batch):
    max_c = max(len(b["content"]) for b in batch)
    max_s = max(len(b["summary"]) for b in batch)
    content = torch.stack([
        torch.cat([b["content"], torch.full((max_c - len(b["content"]),), PAD_ID, dtype=torch.long)])
        for b in batch
    ])
    summary = torch.stack([
        torch.cat([b["summary"], torch.full((max_s - len(b["summary"]),), PAD_ID, dtype=torch.long)])
        for b in batch
    ])
    return {"content": content, "summary": summary}


train_loader = DataLoader(
    SummarizationDataset(train_df), batch_size=CONFIG["batch_size"],
    shuffle=True, collate_fn=collate_fn,
)
val_loader = DataLoader(
    SummarizationDataset(val_df), batch_size=CONFIG["batch_size"],
    shuffle=False, collate_fn=collate_fn,
)


# Model
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lengths):
        embedded = self.dropout(self.embedding(src))
        packed = pack_padded_sequence(embedded, src_lengths,
                                      batch_first=True, enforce_sorted=False)
        packed_out, (hidden, cell) = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True,
                                     total_length=src.size(1))
        return out, hidden, cell


class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.encoder_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.decoder_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, src_mask, encoder_features=None):
        if encoder_features is None:
            encoder_features = self.encoder_proj(encoder_outputs)
        decoder_features = self.decoder_proj(hidden).unsqueeze(1)
        energy = torch.tanh(encoder_features + decoder_features)
        scores = self.v(energy).squeeze(-1)
        scores = scores.masked_fill(~src_mask, torch.finfo(scores.dtype).min)
        return torch.softmax(scores, dim=1)


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim,
                 num_layers, dropout, attention):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.attention = attention
        self.lstm = nn.LSTM(embedding_dim + hidden_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, trg, hidden, cell, encoder_outputs, src_mask, encoder_features):
        embedded = self.dropout(self.embedding(trg))
        attn = self.attention(hidden[-1], encoder_outputs, src_mask, encoder_features)
        context = torch.bmm(attn.unsqueeze(1), encoder_outputs)
        lstm_input = torch.cat((embedded, context), dim=2)
        outputs, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
        predictions = self.fc(torch.cat((outputs, context), dim=2))
        return predictions, hidden, cell, attn


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = src.shape[0], trg.shape[1]
        vocab_size = self.decoder.fc.out_features
        src_lengths = (src != PAD_ID).sum(dim=1).cpu()
        src_mask = src != PAD_ID

        enc_out, hidden, cell = self.encoder(src, src_lengths)
        enc_feat = self.decoder.attention.encoder_proj(enc_out)
        decoder_input = trg[:, 0].unsqueeze(1)
        outputs = torch.zeros(batch_size, trg_len, vocab_size, device=self.device)

        for t in range(1, trg_len):
            out, hidden, cell, _ = self.decoder(
                decoder_input, hidden, cell, enc_out, src_mask, enc_feat,
            )
            outputs[:, t, :] = out.squeeze(1)
            use_teacher = random.random() < teacher_forcing_ratio
            decoder_input = trg[:, t].unsqueeze(1) if use_teacher else out.argmax(2)
        return outputs


encoder   = Encoder(VOCAB_SIZE, CONFIG["embedding_dim"], CONFIG["hidden_dim"],
                    CONFIG["num_layers"], CONFIG["dropout"])
attention = Attention(CONFIG["hidden_dim"])
decoder   = Decoder(VOCAB_SIZE, CONFIG["embedding_dim"], CONFIG["hidden_dim"],
                    CONFIG["num_layers"], CONFIG["dropout"], attention)
model     = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
scaler    = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


# Training
best_val = float("inf")
accum = CONFIG["accum_steps"]

for epoch in range(CONFIG["epochs"]):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                desc=f"Epoch {epoch + 1}/{CONFIG['epochs']}")
    for idx, batch in pbar:
        src = batch["content"].to(device, non_blocking=True)
        trg = batch["summary"].to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                            enabled=(device.type == "cuda")):
            out = model(src, trg, CONFIG["teacher_forcing"])
            loss = criterion(
                out[:, 1:].reshape(-1, VOCAB_SIZE).float(),
                trg[:, 1:].reshape(-1),
            )
            scaled = loss / accum
        train_loss += loss.item()
        scaler.scale(scaled).backward()
        if (idx + 1) % accum == 0 or idx + 1 == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            src = batch["content"].to(device, non_blocking=True)
            trg = batch["summary"].to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=(device.type == "cuda")):
                out = model(src, trg, 1.0)
                loss = criterion(
                    out[:, 1:].reshape(-1, VOCAB_SIZE).float(),
                    trg[:, 1:].reshape(-1),
                )
            val_loss += loss.item()
    val_loss /= len(val_loader)
    tqdm.write(f"Epoch {epoch + 1} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "val_loss": best_val,
        }, CKPT_PATH)


# Reload best checkpoint
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()


# Greedy decoding
def generate_summary(model, text, sp_proc, cfg, device):
    model.eval()
    ids = sp_proc.encode(text, out_type=int)[:cfg["max_content_len"]] or [UNK_ID]
    src         = torch.tensor([ids], dtype=torch.long, device=device)
    src_lengths = torch.tensor([len(ids)], dtype=torch.long)
    src_mask    = src != PAD_ID

    with torch.no_grad():
        enc_out, hidden, cell = model.encoder(src, src_lengths)
        enc_feat = model.decoder.attention.encoder_proj(enc_out)
        dec_in = torch.tensor([[BOS_ID]], dtype=torch.long, device=device)
        out_ids = []
        for _ in range(cfg["max_summary_len"]):
            out, hidden, cell, _ = model.decoder(
                dec_in, hidden, cell, enc_out, src_mask, enc_feat,
            )
            pid = out.argmax(dim=-1).item()
            if pid == EOS_ID:
                break
            if pid not in (BOS_ID, PAD_ID):
                out_ids.append(pid)
            dec_in = torch.tensor([[pid]], dtype=torch.long, device=device)
    return sp_proc.decode(out_ids)


# Batch inference
rows = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Predicting"):
    rows.append({
        "source":    str(row["content"]),
        "reference": str(row["summary"]),
        "predicted": generate_summary(model, str(row["content"]), sp, CONFIG, device),
    })
pred_df = pd.DataFrame(rows)
pred_df.to_csv(PRED_CSV, index=False, encoding="utf-8-sig")


# Evaluation
pred_df = pd.read_csv(PRED_CSV).fillna("")
empty = pred_df["predicted"].astype(str).str.strip() == ""
n_empty = int(empty.sum())
if n_empty:
    pred_df.loc[empty, "predicted"] = "।"

refs = pred_df["reference"].astype(str).tolist()
hyps = pred_df["predicted"].astype(str).tolist()

bleu_f  = BLEU(tokenize="flores200", max_ngram_order=4).corpus_score(hyps, [refs]).score
bleu_13 = BLEU(tokenize="13a",      max_ngram_order=4).corpus_score(hyps, [refs]).score


class NepaliTokenizer:
    def tokenize(self, text):
        return str(text).strip().split()


scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False,
    tokenizer=NepaliTokenizer(),
)
rouge = [scorer.score(r, h) for r, h in zip(refs, hyps)]
r1 = float(np.mean([x["rouge1"].fmeasure for x in rouge]))
r2 = float(np.mean([x["rouge2"].fmeasure for x in rouge]))
rL = float(np.mean([x["rougeL"].fmeasure for x in rouge]))

# BERTScore with bert-base-multilingual-cased as used in the paper
P, R, F1 = bert_score_fn(
    hyps, refs,
    model_type="bert-base-multilingual-cased",
    verbose=False,
    device=device.type,
)


def jaccard_word(a, b):
    A, B = set(str(a).split()), set(str(b).split())
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)


jac = float(np.mean([jaccard_word(h, r) for h, r in zip(hyps, refs)]))

summary = {
    "model":               "Attention LSTM Seq2Seq",
    "sacrebleu_flores200": round(bleu_f,  4),
    "sacrebleu_13a":       round(bleu_13, 4),
    "rouge1":              round(r1, 4),
    "rouge2":              round(r2, 4),
    "rougeL":              round(rL, 4),
    "bertscore_p":         round(P.mean().item(),  4),
    "bertscore_r":         round(R.mean().item(),  4),
    "bertscore_f1":        round(F1.mean().item(), 4),
    "jaccard_word":        round(jac, 4),
    "n_samples":           len(refs),
    "n_empty":             n_empty,
    "best_val_loss":       round(best_val, 4),
}

pd.Series(summary).to_csv(SCORES_CSV)